# Упражнения <a id='tasks'></a>

Необходимо изучить библиотеку [tslearn](https://tslearn.readthedocs.io/en/stable/).

1. Взять выбранные для лабы 14 набор(ы) данных.

3. Попробовать все методы классификации и регрессии, описанные в блоноте 14 aeon:
    Distance-based
    Свертки (модели семейства Rocket и Hydra)
    Feature-based
    Deep Learning

    Для каждого подхода обучть не менее 3х разных моделей для классификации и регрессии.
    
     
    

In [1]:
!pip install tslearn aeon[all_extras]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.9/387.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.7/364.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 7.7 MB/s eta 0:00:00

In [3]:
from tslearn.datasets import UCR_UEA_datasets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from aeon.datasets import load_from_ts_file

X_train, y_train = load_from_ts_file("./dataset/ECG200_TRAIN.ts", return_type="numpy2D")
X_test, y_test = load_from_ts_file("./dataset/ECG200_TEST.ts", return_type="numpy2D")

X_train_ts = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_ts = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

scaler = TimeSeriesScalerMeanVariance()
X_train_ts = scaler.fit_transform(X_train_ts)
X_test_ts = scaler.transform(X_test_ts)

print(f"Данные готовы. Формат для tslearn: {X_train_ts.shape}")

Данные готовы. Формат для tslearn: (100, 96, 1)


In [4]:
from tslearn.neighbors import KNeighborsTimeSeriesClassifier
from tslearn.svm import TimeSeriesSVC

knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=3, metric="dtw")

knn_softdtw = KNeighborsTimeSeriesClassifier(n_neighbors=3, metric="softdtw")

svc_gak = TimeSeriesSVC(kernel="gak", gamma="auto")

models_distance = [("k-NN DTW", knn_dtw), ("k-NN Soft-DTW", knn_softdtw), ("SVC GAK", svc_gak)]

In [7]:
from aeon.classification.convolution_based import RocketClassifier, HydraClassifier, Arsenal

models_conv = [
    ("Rocket", RocketClassifier()),
    ("Hydra", HydraClassifier()),
    ("Arsenal", Arsenal())
]

In [8]:
from aeon.classification.deep_learning import InceptionTimeClassifier, ResNetClassifier
from tslearn.shapelets import LearningShapelets

inception = InceptionTimeClassifier(n_epochs=20, verbose=False)

resnet = ResNetClassifier(n_epochs=20, verbose=False)

shapelets = LearningShapelets(n_shapelets_per_size={10: 5}, max_iter=50, verbose=0)

models_feature = [("InceptionTime", inception), ("ResNet", resnet), ("Learning Shapelets", shapelets)]

In [9]:
from sklearn.metrics import accuracy_score

def evaluate_all(groups, X_tr_ts, y_tr, X_te_ts, y_te, X_tr_2d, X_te_2d):
    for group_name, models in groups:
        print(f"\n--- {group_name} ---")
        for name, model in models:
            try:
                if "Rocket" in name or "Hydra" in name or "Arsenal" in name or "Inception" in name or "ResNet" in name:
                    model.fit(X_tr_2d, y_tr)
                    y_pred = model.predict(X_te_2d)
                else:
                    model.fit(X_tr_ts, y_tr)
                    y_pred = model.predict(X_te_ts)

                acc = accuracy_score(y_te, y_pred)
                print(f"{name:20} | Accuracy: {acc:.4f}")
            except Exception as e:
                print(f"Ошибка в {name}: {e}")

all_test_groups = [
    ("Метрики расстояний (tslearn)", models_distance),
    ("Свертки (Rocket/Hydra)", models_conv),
    ("Deep Learning & Shapelets", models_feature)
]

evaluate_all(all_test_groups, X_train_ts, y_train, X_test_ts, y_test, X_train, X_test)


--- Метрики расстояний (tslearn) ---
k-NN DTW             | Accuracy: 0.8000
k-NN Soft-DTW        | Accuracy: 0.8400
SVC GAK              | Accuracy: 0.8700

--- Свертки (Rocket/Hydra) ---
Rocket               | Accuracy: 0.9100
Hydra                | Accuracy: 0.8900
Arsenal              | Accuracy: 0.9000

--- Deep Learning & Shapelets ---
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 660ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 630ms/step


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 624ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 593ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 854ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 605ms/step
InceptionTime        | Accuracy: 0.5700
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 734ms/step
ResNet               | Accuracy: 0.6400


/usr/local/lib/python3.12/dist-packages/tslearn/shapelets/shapelets.py:498: FutureWarning: The default value for 'scale' is set to False in version 0.4 to ensure backward compatibility, but is likely to change in a future version.
  warnings.warn("The default value for 'scale' is set to False "


Learning Shapelets   | Accuracy: 0.6400
